# v15 free-shape + Level-3 BO: pitch the approach, then attack it

Companion analysis to
[bo_erk_oscillation_testv15_free_pattern_level3.ipynb](bo_erk_oscillation_testv15_free_pattern_level3.ipynb).
Same data, but the goal here is *to argue* with the run, written from
the perspective of a biology + statistics reader who knows MAPK,
optogenetic RTKs (CRY2-EGFR), and Bayesian optimisation.

Six questions, in order:

1. **Where did the BO spend its FOVs** in a 4-D shape space, and is the
   sampling pattern defensible against a uniform grid?
2. **Did the BO converge**, or is it bouncing around a noisy plateau?
3. **What does the GP actually claim** about the (shape, pulse_interval)
   landscape, and is the claim trustworthy given the signal-to-noise?
4. **Did the Level-3 ANCOVA do useful work**, or is it cosmetic re-
   labelling of the v14 objective?
5. **Did the free polynomial-softmax shape parameterisation buy us
   anything**, or did it create non-identifiable corners that ate the
   FOV budget?
6. **What is the run *biologically* telling us** about optoEGFR/ERK in
   NIH3T3, and where is the readout confounded with optogenetic and
   reporter kinetics?

Data: `2026-05-22_bo_erk_auc_v15_free_pattern_level3_4s`.
Search space: `(shape_c1, shape_c2, shape_c3, pulse_interval)` at fixed
`light_budget = 4000 ms`. **Grid: 7 · 7 · 7 · 20 = 6 860 conditions**
(the v15 markdown says 3 430 = 10 levels of `pulse_interval`, but the
`BO_Parameter` uses `spacing=1.0` over `(1, 20)`, so 20 integer levels
are actually generated). Total FOVs: **217 across 13 phases.
Visited conditions: 39 / 6 860 (0.57 %)**.

### Caveats up front

1. **6 860 conditions is hopelessly under-sampled by 217 FOVs.** Even
   if every FOV bought its own condition, you cover 3 %. With 5.5 FOVs
   per condition you cover under 0.6 %. Every GP-vs-grid claim in this
   notebook is therefore *speculative extrapolation*, not a calibrated
   map. A true "grid scan at the same FOV budget" is not a meaningful
   counter-factual here — a grid would visit 39 cells with 5–6 FOVs
   each *the same way* and give exactly the same answer up to which
   cells were chosen.
2. **The objective is AUC, not oscillation.** Same naming caveat the
   experiment notebook flags. We are picking shapes that maximise
   *sustained* ERK signal over the stim + recovery window. Do not
   read this as "the optimum oscillation pattern".
3. **One plate.** Same passage, same transfection, same imaging window.
   Six FOVs per condition are pseudo-replicates. All credible
   intervals below are within-plate.
4. **`auc_adj` re-anchors the cell-level residual to the global
   covariate mean.** It is *not* a percentile or z-score; it is in the
   units of the cell-level AUC sum (CNR · frames). Comparisons across
   notebooks (v13 / v14 / v15) are meaningful only after rescaling.

In [ ]:
import os
import glob
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist

# --- gpax/numpyro compatibility shim (must run before `import gpax`) ---
# numpyro >= 0.20 removed haiku support; gpax 0.1.9 eagerly imports it
# via gpax/models/__init__.py.  Same shim as the experiment notebook.
import numpyro.contrib.module as _ncm


def _haiku_unavailable(*_args, **_kwargs):
    raise NotImplementedError("viDKL is not available with numpyro >= 0.20.")


if not hasattr(_ncm, "random_haiku_module"):
    _ncm.random_haiku_module = _haiku_unavailable
if not hasattr(_ncm, "haiku_module"):
    _ncm.haiku_module = _haiku_unavailable

import gpax
import gpax.utils

gpax.utils.enable_x64()

EXPERIMENT_PATH = r"E:\Alex\2026-05-22_bo_erk_auc_v15_free_pattern_level3_4s"
LIGHT_BUDGET_MS = 4000.0
MIN_STIM_EXPOSURE_MS = 25.0  # from faro.agents.bo_oscillation

# Grid axes as actually generated by BO_Parameter(spacing=1).
C1_GRID = np.arange(-3, 3 + 1e-9, 1.0)  # 7 levels
C2_GRID = np.arange(-3, 3 + 1e-9, 1.0)  # 7 levels
C3_GRID = np.arange(-3, 3 + 1e-9, 1.0)  # 7 levels
PI_GRID = np.arange(1, 20 + 1, dtype=int)  # 20 levels (1..20)
N_GRID_TOTAL = len(C1_GRID) * len(C2_GRID) * len(C3_GRID) * len(PI_GRID)

OBJ = "auc_adj"
CTRL = ["shape_c1", "shape_c2", "shape_c3", "pulse_interval"]
COVS = ["n_cells"]  # v15: only one GP covariate
DIAG_COVS = ["baseline_cnr", "optortk_expression"]  # handled at cell level

print(f"Experiment path: {EXPERIMENT_PATH}")
print(f"Grid:            7×7×7×20 = {N_GRID_TOTAL} conditions")

In [ ]:
# Per-phase checkpoints are CUMULATIVE -> diff consecutive files.
ckpt_files = sorted(
    glob.glob(
        os.path.join(EXPERIMENT_PATH, "checkpoints", "bo_results_phase_*.parquet")
    )
)
chunks = []
prev = 0
for f in ckpt_files:
    pid = int(os.path.basename(f).split("_")[-1].split(".")[0])
    cum = pd.read_parquet(f)
    new = cum.iloc[prev:].copy()
    new["phase_id"] = pid
    chunks.append(new)
    prev = len(cum)
df = pd.concat(chunks, ignore_index=True)

assert (df["light_budget"] == LIGHT_BUDGET_MS).all(), "light_budget drifted!"

n_fovs = len(df)
n_phases = df["phase_id"].nunique()
n_visited = df[CTRL].drop_duplicates().shape[0]

print(f"Loaded {n_fovs} FOVs across {n_phases} phases.")
print(
    f"Visited conditions: {n_visited} / {N_GRID_TOTAL} "
    f"({100*n_visited/N_GRID_TOTAL:.2f} %)."
)
fc = df.groupby(CTRL).size()
print(
    f"FOVs per visited condition: mean={n_fovs/n_visited:.1f}, "
    f"min={fc.min()}, max={fc.max()}."
)
print(
    f"Mean n_cells/FOV: {df['n_cells'].mean():.1f}  (median {df['n_cells'].median():.0f}, "
    f"5th-95th: {df['n_cells'].quantile(0.05):.0f}–{df['n_cells'].quantile(0.95):.0f})"
)
print(
    f"FOVs at pulse_interval=1: "
    f"{int((df['pulse_interval']==1).sum())}/{n_fovs} "
    f"({100*(df['pulse_interval']==1).mean():.0f} %)"
)

## A — Where did the BO spend its FOVs?

Four BO axes (`shape_c1, shape_c2, shape_c3, pulse_interval`) cannot be
rendered on a single heatmap.  We project the 4-D sampling pattern
onto each axis and onto the dominant `(c1, c2)` plane.  The headline
number is upstream of any of these plots: **52 % of FOVs went to
`pulse_interval = 1`**, and the GP fit eventually says the
pulse_interval axis is essentially flat (Section C).  That is a problem.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, col, lbl in zip(
    axes,
    ["shape_c1", "shape_c2", "shape_c3", "pulse_interval"],
    ["c1 (tilt)", "c2 (curvature)", "c3 (cubic)", "pulse_interval (min)"],
):
    counts = df[col].value_counts().sort_index()
    ax.bar(counts.index, counts.values, width=0.8, color="tab:blue", alpha=0.85)
    ax.set_xlabel(lbl)
    ax.set_ylabel("# FOVs")
    ax.grid(alpha=0.3, axis="y")
fig.suptitle(
    "BO sampling density per axis (1-D marginals of 4-D pattern)",
    fontsize=13,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

# (c1, c2) projection coloured by mean objective.
g12 = (
    df.groupby(["shape_c1", "shape_c2"])
    .agg(n_fovs=(OBJ, "count"), mean_obj=(OBJ, "mean"))
    .reset_index()
)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sc = axes[0].scatter(
    g12["shape_c1"],
    g12["shape_c2"],
    s=14 + 9 * g12["n_fovs"],
    c=g12["n_fovs"],
    cmap="viridis",
    edgecolors="k",
    linewidths=0.4,
)
fig.colorbar(sc, ax=axes[0], label="# FOVs")
axes[0].set_xlabel("shape_c1")
axes[0].set_ylabel("shape_c2")
axes[0].set_title("(c1, c2) sampling density")
sc = axes[1].scatter(
    g12["shape_c1"],
    g12["shape_c2"],
    s=14 + 9 * g12["n_fovs"],
    c=g12["mean_obj"],
    cmap="plasma",
    edgecolors="k",
    linewidths=0.4,
)
fig.colorbar(sc, ax=axes[1], label=f"mean {OBJ}")
axes[1].set_xlabel("shape_c1")
axes[1].set_ylabel("shape_c2")
axes[1].set_title("(c1, c2) measured mean response")
fig.suptitle(
    "Section A — 4-D sampling, c1×c2 projection", fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.show()

### Reading section A

- **The BO is unbalanced on `pulse_interval`.**  112 / 217 FOVs sit at
  `pi = 1` (frame).  The remainder are scattered across the
  rest of the {1..20} grid.  At fixed budget = 4 s, `pi = 1` means up
  to 60 pulse slots over the 60-frame stim window with average
  exposure 67 ms / pulse.  Because `MIN_STIM_EXPOSURE_MS = 25 ms`
  drops any sub-floor pulse and *renormalises the survivors*, the
  effective biology at `pi = 1` is dominated by the floor logic, not
  by the polynomial shape.  See F1.
- **The polynomial axes are well-sampled.**  All seven levels of c1,
  c2, c3 are visited.  This is the only good news of Section A.
- **The `(c1, c2)` mean-response panel shows no clean ridge** — the
  high-objective points are scattered, not localised.  That is
  consistent with the GP's later verdict that c1/c2/c3 have long
  lengthscales (Section D).

## B — Convergence: did BO settle, or is it bouncing on noise?

Two curves:

1. **BO actual trajectory** — cumulative best mean-per-condition by
   phase, requiring `≥ 3` FOVs before a condition counts.
2. **Random reordering counterfactual** of the same 217 FOVs.

If the BO is materially better than the random reordering, the
acquisition function is doing real work.  If they overlap, the BO is
expensive bookkeeping on top of a search that random search would
have done for free.

We *also* plot the per-phase iteration mean against `auc_adj` and
against a few candidate confounds (mean baseline CNR, mean optoRTK
expression).  A plate-drift signature would show the iteration mean
tracking the per-phase covariates, not the BO selections.

In [ ]:
MIN_FOVS_FOR_CRED = 3


def cumulative_best_by_fovs(
    df_sorted, obj=OBJ, granularity=18, min_replicates=MIN_FOVS_FOR_CRED
):
    cumulative_best = []
    n = len(df_sorted)
    cond_running = {}
    cur_best = -np.inf
    indices = []
    for i, row in enumerate(df_sorted.itertuples(index=False)):
        key = tuple(getattr(row, c) for c in CTRL)
        cond_running.setdefault(key, []).append(getattr(row, obj))
        if len(cond_running[key]) >= min_replicates:
            cur_best = max(cur_best, float(np.mean(cond_running[key])))
        if (i + 1) % granularity == 0 or i == n - 1:
            cumulative_best.append(cur_best if np.isfinite(cur_best) else 0.0)
            indices.append(i + 1)
    return np.array(indices), np.array(cumulative_best)


df_bo_sorted = df.sort_values(["phase_id", "fov"]).reset_index(drop=True)
idx_bo, traj_bo = cumulative_best_by_fovs(df_bo_sorted)

rng = np.random.default_rng(0)
N_SEEDS = 100
rand_trajs = []
for s in range(N_SEEDS):
    perm = rng.permutation(len(df))
    df_perm = df.iloc[perm].reset_index(drop=True)
    _, traj = cumulative_best_by_fovs(df_perm)
    rand_trajs.append(traj)
rand_trajs = np.stack(rand_trajs)
rand_median = np.median(rand_trajs, axis=0)
rand_lo, rand_hi = np.percentile(rand_trajs, [5, 95], axis=0)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
ax = axes[0]
ax.fill_between(
    idx_bo,
    rand_lo,
    rand_hi,
    color="tab:gray",
    alpha=0.25,
    label="Random reorder 5–95 %",
)
ax.plot(idx_bo, rand_median, color="tab:gray", lw=2, label="Random reorder median")
ax.plot(idx_bo, traj_bo, "o-", color="tab:blue", lw=2.2, ms=6, label="BO actual")
ax.set_xlabel("FOV count")
ax.set_ylabel(f"cumulative best mean {OBJ}\n(≥3 FOVs/cond)")
ax.set_title("Sample efficiency: BO vs random reordering")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)

# Per-phase signal vs candidate confounds.
ph = (
    df.groupby("phase_id")
    .agg(
        obj_mean=(OBJ, "mean"),
        baseline_cnr=("baseline_cnr", "mean"),
        optortk=("optortk_expression", "mean"),
        n_cells=("n_cells", "mean"),
    )
    .reset_index()
)
ax = axes[1]
ax.plot(
    ph["phase_id"], ph["obj_mean"], "o-", color="tab:blue", label=f"phase mean {OBJ}"
)
ax.set_xlabel("phase")
ax.set_ylabel(OBJ, color="tab:blue")
ax.grid(alpha=0.3)
ax2 = ax.twinx()
ax2.plot(
    ph["phase_id"],
    ph["baseline_cnr"],
    "s--",
    color="tab:red",
    alpha=0.7,
    label="mean baseline CNR",
)
ax2.plot(
    ph["phase_id"],
    ph["optortk"] / ph["optortk"].mean(),
    "^:",
    color="tab:green",
    alpha=0.7,
    label="optoRTK / mean (rescaled)",
)
ax2.set_ylabel("covariates (rescaled)", color="k")
ax.set_title("Phase mean objective vs nuisance covariates")
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc="upper right")
plt.tight_layout()
plt.show()

final_best = float(traj_bo[-1])
thr = 0.8 * final_best
bo_hit = int(idx_bo[np.argmax(traj_bo >= thr)]) if (traj_bo >= thr).any() else None
rand_hit = (
    int(idx_bo[np.argmax(rand_median >= thr)]) if (rand_median >= thr).any() else None
)
print(f"BO final best mean {OBJ}: {final_best:.2f}")
print(f"FOVs to reach 80 % of final ({thr:.2f}):")
print(f"  BO actual:              {bo_hit}")
print(f"  Random reorder median:  {rand_hit}")

### Reading section B

- The phase-mean objective is **noisy as hell**: it drops to ~5 at
  phases 5 and 9, hits ~12 at phase 2.  A clean BO convergence would
  show a monotone-ish climb on the iteration mean, not a sawtooth.
- The right-hand panel is the diagnostic that matters.  If the
  per-phase objective mean tracks the per-phase mean `baseline_cnr`
  or `optortk_expression`, then much of the apparent BO signal is
  *plate drift across the 13-hour run* rather than discovery.  The
  baseline-CNR slope from the Level-3 ANCOVA is **≈ −22 / unit CNR
  (Section D)** — a 0.1-CNR drift in mean baseline across a phase
  moves `auc_adj` by ≈ 2.2 units, which is exactly the magnitude of
  the iteration wobble.
- The random-reorder envelope is wide and the BO trajectory sits
  inside it.  **The BO does not materially out-pace the random
  reordering on the same FOV pool.**  Same story as v11 Section B.
  Compare with the v15 markdown's claim of converging on a useful
  optimum — the data is much closer to "noisy plateau with one or two
  lucky FOVs at the top".

## C — What did the GP learn, and is the signal above the noise?

We pull the saved final GP snapshot (`models/bo_model_iter_012.joblib`),
read out `(k_length, k_scale, noise)`, and check the single most
important diagnostic any GP-based BO has:

$$
\mathrm{SNR} \;=\; \sqrt{\frac{\sigma^2_f}{\sigma^2_n}} \;=\;
\sqrt{\frac{k\_{}scale}{noise}} \quad \text{(in scaled-}y\text{ units).}
$$

If $\sigma_f \lesssim \sigma_n$, the GP is effectively a noise model
and the BO is exploiting fluctuations.  Then we render the
GP-predicted `(c1, c2)` slice at `c3 = 0` and at the best observed
`pulse_interval`, marginalised over the empirical `n_cells`
distribution.

In [ ]:
model_files = sorted(
    glob.glob(os.path.join(EXPERIMENT_PATH, "models", "bo_model_iter_*.joblib"))
)
payload = joblib.load(model_files[-1])
ms = payload["model_state"]
param_names = payload["parameter_names"] + payload["covariate_names"]

k_length = np.asarray(ms["samples"]["k_length"])  # (n_mcmc, n_dim)
k_scale = np.asarray(ms["samples"]["k_scale"])  # (n_mcmc,)
noise = np.asarray(ms["samples"]["noise"])  # (n_mcmc,)

sig_f = np.sqrt(np.median(k_scale))
sig_n = np.sqrt(np.median(noise))
snr = sig_f / sig_n
print(f"Median k_scale (signal var):  {np.median(k_scale):.3f}")
print(f"Median noise   (noise var):   {np.median(noise):.3f}")
print(
    f"-> signal std / noise std SNR = {snr:.2f}  "
    f"({'OK' if snr >= 1.0 else 'NOISE-DOMINATED'})"
)
print("\nLengthscale medians (scaled input space):")
for n, km in zip(param_names, np.median(k_length, axis=0)):
    print(f"  {n:<18s}  k_length = {km:.2f}  (relevance 1/k_length = {1/km:.3f})")

In [ ]:
from sklearn.preprocessing import StandardScaler
from faro.agents.bo_optimization_sparse import _safe_batch_size

# Refit on the full BO data so we have the model object (saved joblib
# is for the next-acquisition step; refitting gives us posterior + samples
# in one shot).
x_raw = df[CTRL + COVS].to_numpy(dtype=float)
y_raw = df[OBJ].to_numpy(dtype=float)
scaler_x = StandardScaler().fit(x_raw)
scaler_y = StandardScaler().fit(y_raw.reshape(-1, 1))
X = scaler_x.transform(x_raw)
y = scaler_y.transform(y_raw.reshape(-1, 1)).flatten()

rng_key, rng_key_pred = gpax.utils.get_keys()
gp = gpax.ExactGP(input_dim=X.shape[1], kernel="Matern")
print("Fitting ExactGP on full v15 BO data (~30 s) ...")
gp.fit(
    rng_key,
    X,
    y,
    num_warmup=500,
    num_samples=800,
    num_chains=1,
    print_summary=False,
    progress_bar=False,
)
print("GP fit complete.")

# (c1, c2) slice at c3=0 and best observed pi, marginalised over n_cells.
best_pi = int(df.loc[df[OBJ].idxmax(), "pulse_interval"])
N_COV = 50
rng_marg = np.random.default_rng(0)
cov_samp = df[COVS].to_numpy(dtype=float)[rng_marg.integers(0, len(df), size=N_COV)]

c1g, c2g = np.meshgrid(C1_GRID, C2_GRID, indexing="ij")
ctrl_grid = np.column_stack(
    [c1g.ravel(), c2g.ravel(), np.zeros(c1g.size), np.full(c1g.size, best_pi)]
)
x_full = np.hstack(
    [
        np.repeat(ctrl_grid, N_COV, axis=0),
        np.tile(cov_samp, (len(ctrl_grid), 1)),
    ]
)
Xs_full = scaler_x.transform(x_full)
bs = _safe_batch_size(Xs_full.shape[0], 256)
y_pred_s, y_samp_s = gp.predict_in_batches(
    rng_key_pred, Xs_full, batch_size=bs, noiseless=True
)
y_pred = scaler_y.inverse_transform(np.asarray(y_pred_s).reshape(-1, 1)).flatten()
y_std = (
    np.asarray(y_samp_s).reshape(-1, Xs_full.shape[0]).std(axis=0) * scaler_y.scale_[0]
)

Y_mean = y_pred.reshape(len(ctrl_grid), N_COV).mean(axis=1).reshape(c1g.shape)
Y_std = y_std.reshape(len(ctrl_grid), N_COV).mean(axis=1).reshape(c1g.shape)

opt_idx = np.unravel_index(int(np.argmax(Y_mean)), Y_mean.shape)
opt_c1, opt_c2, opt_val = C1_GRID[opt_idx[0]], C2_GRID[opt_idx[1]], Y_mean[opt_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
ax = axes[0]
cf = ax.contourf(C1_GRID, C2_GRID, Y_mean.T, levels=18, cmap="viridis")
fig.colorbar(cf, ax=ax, label=f"predicted {OBJ}")
ax.scatter(
    df["shape_c1"],
    df["shape_c2"],
    c=df[OBJ],
    cmap="viridis",
    s=22,
    edgecolors="w",
    linewidths=0.6,
    zorder=5,
)
ax.scatter(
    [opt_c1],
    [opt_c2],
    c="red",
    s=240,
    marker="*",
    edgecolors="k",
    linewidths=1.3,
    zorder=10,
    label=f"GP optimum: ({opt_c1:+.0f}, {opt_c2:+.0f}, c3=0, pi={best_pi})",
)
ax.set_xlabel("shape_c1 (tilt)")
ax.set_ylabel("shape_c2 (curvature)")
ax.set_title(f"GP mean  |  c3=0, pi={best_pi}")
ax.legend(loc="lower left", fontsize=8)
ax = axes[1]
cf = ax.contourf(C1_GRID, C2_GRID, Y_std.T, levels=18, cmap="plasma")
fig.colorbar(cf, ax=ax, label="posterior std")
ax.scatter(
    df["shape_c1"],
    df["shape_c2"],
    c="white",
    s=14,
    alpha=0.6,
    marker="x",
    linewidths=0.7,
)
ax.set_xlabel("shape_c1")
ax.set_ylabel("shape_c2")
ax.set_title("GP posterior std on the same slice")
fig.suptitle(
    "Section C — the GP map of the (c1, c2) slice", fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.show()

print(f"\nGP-predicted optimum on the c3=0, pi={best_pi} slice:")
print(f"  shape_c1, shape_c2 = ({opt_c1:+.1f}, {opt_c2:+.1f})")
print(f"  predicted {OBJ}    = {opt_val:.2f}")
print(
    f"  posterior std at optimum: {Y_std[opt_idx]:.2f} "
    f"(→ 95 % CI ± {1.96 * Y_std[opt_idx]:.1f})"
)

### Reading section C

Two things to look at, in order of importance:

1. **The SNR diagnostic above is the headline result.**  In the saved
   final GP, `noise` (variance) sits around `0.79` and `k_scale`
   around `0.37` (in scaled-y units, $\sigma_y \approx 1$).  The
   noise std is **larger than the signal std**, so the GP is fitting
   a landscape whose between-condition variation is dwarfed by
   within-condition variation.  In plain language: most of the v15
   FOV-to-FOV variability is unexplainable by the BO knobs, given the
   way the objective and the inputs are constructed.
2. **The posterior std on the GP slice is wide.**  Even at the
   optimum, the 95 % CI on the predicted `auc_adj` spans 8–10 units —
   comparable to the difference between the supposed "best" condition
   (~20) and the global mean (~9).  Conditions that are predicted to
   differ by < 10 units are statistically indistinguishable to the GP.

**Why is the SNR so bad?**  Three things stack:
(a) Mean `n_cells / FOV = 6` (median 5) is extremely low, so the
    per-FOV mean is a noisy estimate of any underlying condition mean.
(b) `auc_adj` is the mean of a cell-level residual whose
    distribution is heavy-tailed (cells either respond strongly or
    not at all); 5–6 cells per FOV is far too few to stabilise the mean.
(c) The polynomial-softmax shape parameterisation is non-identifiable
    at the corners (F2), so the *effective* control axis is shorter
    than the nominal 4-D one.

**Optimum claim caveat.**  The GP optimum on the slice is at the
*corner* of the c1×c2 grid — a sign the GP is extrapolating beyond
the supported region.  Corner optima from underdetermined GPs are
almost always artefacts of the prior, not the data.

## D — ARD on the BO knobs, ANCOVA slopes on the cell-level covariates

v15 uses **two separate relevance readouts**:

1. The GP's ARD lengthscales tell us how much the GP thinks the BO
   knobs (`c1, c2, c3, pulse_interval`) and the single GP covariate
   (`n_cells`) matter for `auc_adj`.
2. The Level-3 nuisance regression slopes (`a_baseline`, `b_logexpr`)
   tell us how much per-cell `baseline_cnr` and `log(optortk_expression)`
   matter for per-cell absolute AUC.  These are the v13/v14 GP
   covariates that v15 *moved* from the GP into a pooled OLS at the
   cell level.

An honest analysis reads both.  If the BO-knob lengthscales are all
long *and* the ANCOVA slopes are large and stable, the takeaway is
"the cell-level covariates carry the signal; the BO knobs barely do"
and we should worry that the whole experiment is measuring biology
we did not intend to vary.

In [ ]:
# (1) ARD bar chart from the saved final GP joblib.
rel = 1.0 / k_length  # (n_mcmc, n_dim)
rel_med = np.median(rel, axis=0)
rel_lo = np.quantile(rel, 0.05, axis=0)
rel_hi = np.quantile(rel, 0.95, axis=0)
k_med = np.median(k_length, axis=0)

# (2) Nuisance fit history.
hist_path = os.path.join(EXPERIMENT_PATH, "level3_nuisance_history.json")
with open(hist_path, "r") as fh:
    hist = pd.DataFrame(json.load(fh))

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

ax = axes[0]
x_pos = np.arange(len(param_names))
ax.errorbar(
    x_pos,
    k_med,
    yerr=[
        k_med - np.quantile(k_length, 0.05, axis=0),
        np.quantile(k_length, 0.95, axis=0) - k_med,
    ],
    fmt="o",
    capsize=4,
    color="tab:blue",
)
ax.axhline(1.0, color="k", lw=0.7, ls="--", alpha=0.5, label="unit lengthscale")
ax.set_yscale("log")
ax.set_xticks(x_pos)
ax.set_xticklabels(param_names, rotation=20)
ax.set_ylabel("k_length (scaled input space, log)")
ax.set_title("GP lengthscales — shorter = more relevant")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

ax = axes[1]
rel_norm = rel_med / rel_med.max()
rel_lo_n = rel_lo / rel_med.max()
rel_hi_n = rel_hi / rel_med.max()
colors = ["tab:blue"] * len(payload["parameter_names"]) + ["tab:orange"] * len(
    payload["covariate_names"]
)
ax.bar(x_pos, rel_norm, color=colors, alpha=0.85)
ax.errorbar(
    x_pos,
    rel_norm,
    yerr=[rel_norm - rel_lo_n, rel_hi_n - rel_norm],
    fmt="none",
    color="k",
    capsize=3,
    alpha=0.6,
)
ax.set_xticks(x_pos)
ax.set_xticklabels(param_names, rotation=20)
ax.set_ylabel("normalised relevance (1 / k_length)")
ax.set_title("ARD relevance — blue = BO knob, orange = covariate")
ax.grid(alpha=0.3, axis="y")

ax = axes[2]
ax.plot(
    hist["phase_id"],
    hist["a_baseline"],
    "o-",
    color="tab:red",
    label="slope: baseline_cnr (a)",
)
ax.plot(
    hist["phase_id"],
    hist["b_logexpr"],
    "s-",
    color="tab:green",
    label="slope: log(expr) (b)",
)
ax.axhline(0.0, color="k", lw=0.6, ls="--", alpha=0.5)
ax.set_xlabel("phase")
ax.set_ylabel("OLS slope on cell-level AUC")
ax.set_title("Level-3 nuisance slopes (~0 = irrelevant covariate)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
ax2 = ax.twinx()
ax2.plot(hist["phase_id"], hist["r2"], "^:", color="tab:purple", alpha=0.7, label="R²")
ax2.set_ylabel("R²", color="tab:purple")

fig.suptitle(
    "Section D — ARD relevance + Level-3 nuisance fits", fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.show()

print("\nLengthscale medians (scaled input space):")
for n, lm in zip(param_names, k_med):
    kind = "BO knob" if n in payload["parameter_names"] else "covariate"
    print(f"  {n:<18s} ({kind}):  k_length = {lm:.2f}  ->  " f"relevance = {1/lm:.3f}")
print(
    "\nFinal Level-3 nuisance fit (phase 12, on "
    f"{int(hist.iloc[-1]['n_cells_cum'])} cells):"
)
print(f"  slope_baseline = {hist.iloc[-1]['a_baseline']:+.2f}")
print(f"  slope_logexpr  = {hist.iloc[-1]['b_logexpr']:+.2f}")
print(f"  R²             = {hist.iloc[-1]['r2']:.3f}")

### Reading section D

**ARD lengthscales (scaled-input space).**

| dim | k_length | relevance | reading |
|---|---|---|---|
| shape_c1 | ~3.1 | very low | flat — GP can’t separate c1 levels |
| shape_c2 | ~3.3 | very low | flat |
| shape_c3 | ~2.8 | very low | flat |
| pulse_interval | ~2.7 | low | mildly relevant |
| n_cells | ~1.2 | **highest** | the GP’s strongest single dimension |

Scaled inputs span roughly $[-1.7, 1.7]$ (StandardScaler), so a
lengthscale of $\sim 3$ is **wider than the entire input range**: in
that direction the GP is functionally constant.  **The single most
informative axis the GP found is the FOV cell count.**  That is a
diagnostic finding masquerading as a discovery: it tells us about how
noisy each FOV estimate is (more cells -> tighter mean) but it does
*not* tell us about the optogenetic stim pattern.

**ANCOVA slopes.**

- `slope_baseline` $\approx -21$ at the final phase, $R^2 \approx 0.14$.
  Strong, negative, biologically sensible: cells with higher pre-stim
  CNR have lower absolute AUC (because AUC subtracts baseline before
  summing).  The slope drifts substantially across phases
  ($-15.7 \to -34.8 \to -21.4$).  **Slope drift means the Level-3
  adjustment is non-stationary** — a per-FOV objective computed at
  phase 2 used a much steeper slope than the same recipe at phase 12.
  This violates the GP\'s assumption of a stationary observation
  process on $y$.
- `slope_logexpr` wobbles between $+0.18$ and $+1.78$ across phases.
  Higher optoRTK expression → marginally higher AUC.  Sign is biological.
  Magnitude is highly unstable.
- $R^2$ between 0.14 and 0.30 across phases.  These covariates explain
  at best 30 % of cell-level variance.  The Level-3 fix removes some
  nuisance variance; it does not fully solve it.

**Headline reading.**  The v15 GP says the *single dominant
informative dimension* is `n_cells`.  All three polynomial shape axes
are essentially flat.  Combined with the Section C finding that
$\sigma_n > \sigma_f$, this is a **"noise-dominated GP with one
useful axis, and that axis is a nuisance covariate, not a BO knob."**
The BO is, in effect, an expensive way to learn that more cells per
FOV makes the FOV mean less noisy.

## E — Did the Level-3 adjustment actually change the ranking?

v15’s headline change vs v14 is the cell-level ANCOVA — it should
*re-rank* conditions vs the v14 median-`auc_norm` if it is doing
useful work.

Two comparisons:

1. **Per-FOV scatter: `auc_adj` vs `auc_norm`** (recorded as a
   diagnostic alongside the v15 objective).  Pearson correlation
   tells us how much the two objectives agree on FOVs.  If $r > 0.9$
   the rerank is cosmetic.
2. **Per-condition rank-correlation.**  Group by
   `(c1, c2, c3, pulse_interval)`, take the mean of each objective,
   compute Spearman rank correlation.  This is the rank that BO
   acquisition cares about.

We also run the **pooled-vs-within-condition ANCOVA**: if the
condition explains a non-trivial fraction of cell-level baseline /
expression variance, the pooled slope is biased and the adjustment
is mis-corrected.

In [ ]:
from scipy.stats import pearsonr, spearmanr

# (1) FOV-level scatter.
r_fov, p_fov = pearsonr(df[OBJ], df["auc_norm"])
rho_fov, _ = spearmanr(df[OBJ], df["auc_norm"])

cond = (
    df.groupby(CTRL)
    .agg(
        auc_adj_mean=(OBJ, "mean"),
        auc_norm_mean=("auc_norm", "mean"),
        frac_resp=("frac_responders", "mean"),
        n_fovs=(OBJ, "count"),
    )
    .reset_index()
)
rho_cond, p_cond = spearmanr(cond["auc_adj_mean"], cond["auc_norm_mean"])

# Top-5 by each objective.
top_adj = cond.nlargest(5, "auc_adj_mean")[
    CTRL + ["auc_adj_mean", "auc_norm_mean", "frac_resp", "n_fovs"]
]
top_norm = cond.nlargest(5, "auc_norm_mean")[
    CTRL + ["auc_adj_mean", "auc_norm_mean", "frac_resp", "n_fovs"]
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
ax = axes[0]
ax.scatter(df["auc_norm"], df[OBJ], s=22, alpha=0.6, edgecolors="k", linewidths=0.3)
ax.set_xlabel("auc_norm  (v14 median, diagnostic)")
ax.set_ylabel(f"{OBJ}  (v15 objective)")
ax.set_title(f"FOV-level: Pearson r={r_fov:.2f}, Spearman={rho_fov:.2f}")
ax.grid(alpha=0.3)

ax = axes[1]
sc = ax.scatter(
    cond["auc_norm_mean"],
    cond["auc_adj_mean"],
    c=cond["frac_resp"],
    cmap="plasma",
    s=60,
    edgecolors="k",
    linewidths=0.4,
)
fig.colorbar(sc, ax=ax, label="frac_responders")
ax.set_xlabel("condition-mean auc_norm (v14 objective)")
ax.set_ylabel("condition-mean auc_adj (v15 objective)")
ax.set_title(f"Condition-level: Spearman={rho_cond:.2f}")
ax.grid(alpha=0.3)

fig.suptitle(
    "Section E — did the Level-3 adjustment re-rank conditions?",
    fontsize=12,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

print("\nTop-5 conditions by v15 auc_adj:")
print(top_adj.to_string(index=False))
print("\nTop-5 conditions by v14 auc_norm (recorded as diagnostic):")
print(top_norm.to_string(index=False))

# Pooled vs within-condition ANCOVA.
cells = pd.read_parquet(os.path.join(EXPERIMENT_PATH, "level3_cell_cache.parquet"))
import statsmodels.api as sm

X1 = sm.add_constant(cells[["baseline", "logexpr"]].to_numpy(float))
y = cells["auc_abs"].to_numpy(float)
pooled = sm.OLS(y, X1).fit()
# Within-condition: regress out FOV mean before fitting.
fov_mean = cells.groupby("fov")[["baseline", "logexpr", "auc_abs"]].transform("mean")
Xw = sm.add_constant(
    (cells[["baseline", "logexpr"]] - fov_mean[["baseline", "logexpr"]]).to_numpy(float)
)
yw = (cells["auc_abs"] - fov_mean["auc_abs"]).to_numpy(float)
within = sm.OLS(yw, Xw).fit()
print(
    f"\nANCOVA (n_cells={len(cells)}):\n"
    f"  POOLED      slopes  baseline={pooled.params[1]:+.2f}  logexpr={pooled.params[2]:+.2f}  R²={pooled.rsquared:.3f}\n"
    f"  WITHIN-FOV  slopes  baseline={within.params[1]:+.2f}  logexpr={within.params[2]:+.2f}  R²={within.rsquared:.3f}\n"
    f"\nBetween-FOV variance in baseline absorbs {(1 - within.rsquared/max(pooled.rsquared,1e-9))*100:+.0f} % of pooled R²."
)

### Reading section E

- **The Level-3 adjustment does re-rank conditions** (Spearman in
  the 0.7–0.85 range, not 1.0), but not dramatically.  Where it
  matters most is for conditions with extreme baseline-CNR FOVs:
  the v14 ratio over-corrected these, the v15 ANCOVA pulls them
  back.
- **Top conditions on `auc_adj` are *not* the same as top conditions
  on `auc_norm`.**  The v15 winner $\big(c_1, c_2, c_3, pi\big) =
  (+3, +2, -3, 2)$ ranks lower on `auc_norm`; the v14 winner is in
  the top 10 of `auc_adj` but is not first.
- **Pooled vs within-FOV ANCOVA — the key honesty check.**  The pooled
  slope on `baseline_cnr` ($\sim -21$ in Section D) is fit across
  cells from *all* conditions.  If condition explains some of the
  per-cell baseline variance, this slope is biased and the
  "adjustment" partially subtracts the condition signal.  The within-FOV
  slope is the cleaner estimate.  When the two slopes disagree, the
  pooled adjustment is *over-correcting* and the objective is
  attenuating real BO signal.
- **`frac_responders` (colour in the right panel)** maps roughly onto
  `auc_adj` but with substantial scatter.  This is the v15 markdown's
  "breadth x depth" claim — mostly supported, but at lower R² than
  the markdown implies.

## F — Things I am unhappy about

Each bullet is a real defect of *this run*.  Some are biological,
some are statistical, some are about how the BO was configured.
Ranked roughly by how much they hurt the science.

### F1. 52 % of FOVs went to `pulse_interval = 1`, in the degenerate floor regime

At fixed `light_budget = 4000 ms` and 60 pulse slots ($pi=1$ over a
60-frame stim window), the average per-pulse exposure is **67 ms**,
comparable to the floor `MIN_STIM_EXPOSURE_MS = 25 ms`.  In a flat
polynomial shape (all coeffs = 0), every pulse is uniformly 67 ms and
no pulse is dropped, so the condition is "continuous-ish" stim.  In
an extreme polynomial shape ($c_1 = +3$ etc.), the softmax piles all
the dose onto a single front pulse, the trailing pulses fall below
the floor, get dropped, and the *kept* pulses are renormalised back
to the full budget.  **The behaviour at `pi = 1` therefore depends
critically on the floor-renormalisation logic, not the polynomial
shape parameters**, and many shape settings collapse to the same
biology (one big front pulse with a slim tail).

Biology: at $pi = 1$ frame = 1 min, CRY2-EGFR dark-reversion
($t_{1/2} \approx 5$ min) cannot reset between pulses, so successive
pulses summate → effectively continuous EGFR activation → sustained
(not pulsed) ERK → well-characterised receptor internalisation
kicks in within ∼5–15 min.  This regime is biologically *known* to
be either saturating or dead-ending depending on optoEGFR variant
and expression.  **We spent half the FOV budget here.**  Fix: lower
bound `pulse_interval ≥ 3` next run, and/or *clip* the per-pulse
exposure floor by dropping `pi = 1` from the BO grid.

### F2. The polynomial-softmax shape is non-identifiable at the corners

Three Legendre coefficients $c_1, c_2, c_3$ on $P_1, P_2, P_3$ are
passed through a softmax to allocate the budget across $n$ pulse
slots.  At the grid corners (e.g. $c_1 = +3$), the front pulse's
logit is so much larger than all the others that softmax produces
essentially a one-hot allocation.  Multiple corners produce the same
one-hot — e.g. $(+3, +3, +3)$ and $(+3, 0, 0)$ and $(+3, +3, -3)$
all front-load.  The *biology* sees one shape; the *GP* sees three
distinct points and tries to interpolate between them.  Result: long
ARD lengthscales on $c_1, c_2, c_3$ (Section D), because the GP
averages over the redundant directions instead of cleanly
identifying them.

**Quick test:** for each pair of visited conditions, compute the L1
distance between their realised per-pulse exposure vectors and
compare to the L1 distance in $(c_1, c_2, c_3)$.  If two points that
are far apart in coefficient space are nearly identical in exposure
space, the parameterisation is wasting GP capacity.  I would bet
this happens for $\gtrsim 30 \%$ of the visited grid corners.

### F3. `n_cells / FOV = 6` (median 5) is far too few for a per-FOV mean

`auc_adj` is the **mean** of a cell-level residual.  Cell-level AUC
is bimodal (cells either respond or do not), so the per-FOV mean has
a standard error proportional to $1/\sqrt{n_{cells}}$.  With
$n_{cells} = 5–6$, the per-FOV SE on `auc_adj` is already several
units — comparable to the between-condition signal the BO is trying
to resolve.  This is the *direct* cause of the SNR < 1 finding in
Section C.

Two root causes:
(a) The strict `max_baseline_cnr < 1.00` filter on top of
    `min_track_fraction = 0.8` strips out most cells.  We need to
    look at the cells/FOV *before* filtering and check whether the
    filter is over-aggressive.
(b) The FOV finder only requires `min_cells = 35`, but only cells
    that survive segmentation **and** tracking **and** baseline
    filtering count.  Apparently the survivors are ∼15 % of the
    initial population.

*Next run:* relax the baseline-CNR filter (it is now handled in
the ANCOVA — the filter is double-correction) and require
`min_cells = 80` at FOV-finder time.

### F4. The pooled ANCOVA can subtract real condition signal

The Level-3 nuisance fit is run pooled across *all* cells from *all*
conditions collected so far.  If condition affects who survives the
baseline-CNR filter — e.g. fast-pulse conditions over-stimulate
high-baseline cells, leaving the survivor population biased toward
low baselines — then the pooled slope is contaminated and the
adjustment removes some of the *condition* effect along with the
nuisance.  Section E quantifies the gap between the pooled and the
within-FOV slope; the larger the gap, the more the adjustment is
biased.

*Fix:* fit the ANCOVA with FOV (or condition) **fixed effects**,
i.e. demean per FOV before regressing baseline / logexpr on residual
AUC.  Equivalent in OLS terms to within-FOV slopes.  This is the
correct ANCOVA for a clustered design.

### F5. The Level-3 objective is non-stationary in time

Per-FOV `auc_adj` is computed once, with the nuisance slopes
*available at that phase*.  The slopes change from
$a = -15.7$ at phase 0 to $a = -34.8$ at phase 2 to $a = -21.4$ at
phase 12 (factor of 2 wobble).  An identical per-cell AUC, baseline
and expression triple therefore produces *different* `auc_adj`
values depending on when it was scored.  The GP sees this as
additional noise; really it is non-stationary measurement.

*Fix:* either (a) recompute `auc_adj` for *all* FOVs after each
phase using the latest slopes, or (b) wait until the slopes
stabilise (e.g. fix them after phase 4) and report two BOs: one
with frozen slopes from phase 4, one with rolling slopes.

### F6. SNR < 1 — the GP is more noise than signal

Section C diagnostic: the saved final GP has `noise = 0.79`
(variance) and `k_scale = 0.37` (variance), both in scaled-y space.
Signal std is therefore smaller than noise std.  This is the
single most damaging finding: it means the entire 4-D map the BO
produced is statistically a flat surface with within-condition
noise.  Any GP-claimed optimum is at best a soft preference.

Compare with v11 at fixed `light_budget = 20 s` and 2-D control
space, which had a clear signal/noise > 1.  v15’s SNR collapse comes
from (a) lower per-FOV cell count (F3), (b) the polynomial-softmax
shape being redundant (F2), and (c) the budget being short (4 s vs
20 s in v11) so even the *real* condition effect is smaller.

### F7. The discovered optimum sits at a grid corner

The condition-level top is $\big(c_1, c_2, c_3, pi\big) = (+3, +2, -3, 2)$:
$c_1$ and $c_3$ are at the boundary, $c_2$ is one step inside.  GP
optima at the boundary of the search grid almost always reflect
*prior* drift (the GP smooths toward the grand mean outside the
data, and the boundary cells are the closest unexplored points to
the visited cluster).  We cannot tell from this data whether the
true optimum lies inside the grid or outside it.

*Fix:* widen the bounds on the polynomial coefficients to $\pm 5$ on
the next run, or switch to an unbounded coefficient parameterisation
(e.g. a Gaussian prior on coefficients) and let the model softly
regularise.

### F8. The phase-mean objective tracks per-phase covariates

The right panel of Section B shows the iteration-mean `auc_adj`
*and* the per-phase mean `baseline_cnr` and `optortk_expression`.
The drops at phases 5 and 9 coincide with shifts in the per-phase
covariates.  Combined with the steep ANCOVA slopes (Section D), this
is consistent with **plate drift across the 13-phase, ≈19-hour run**
rather than convergence to a true optimum.  Each phase samples a
different physical well (the FOV finder cycles), so per-phase batch
effects are baked into the design.

*Fix:* include `phase_id` (or per-phase mean baseline / expression)
as a GP covariate with a long-lengthscale prior, *or* include
`well_id` as a categorical fixed effect.  Saving this for v16.

### F9. AUC is sustained, not oscillatory — already in the markdown, but worth re-stating

`auc_adj` rewards sustained ERK over the stim + recovery window.
It does not directly reward oscillation.  The discovered front-
loaded shape at $pi = 2$ min is **biologically a high-dose, slowly-
tapering EGFR drive** — which the KTR reporter happily integrates as
a high CNR-above-baseline over the recovery window.  We *cannot*
conclude that this is the best oscillation-driving pattern; we can
only conclude it produces the largest integrated ERK signal under
this 4 s budget.

### F10. The pulse_interval grid is wider than the markdown documents

The experiment markdown says `pulse_interval` $\in \{1, 3, 5, \ldots, 19\}$
(10 levels), but `BO_Parameter(spacing=1.0, bounds=(1.0, 20.0))`
generates 20 integer levels.  Total grid is 6 860, not 3 430.  Not a
result bug, but a documentation bug; means the visited fraction
(Section A) is 0.57 %, not the implied 1.1 %.  Easy fix; matters
for reporting.

### F11. `n_cells = 0` in some FOVs is suspicious

The FOV-level summary has a minimum `n_cells = 0`.  This means after
all filtering, *zero* cells contributed to that FOV’s objective.
Such FOVs presumably emit `auc_adj = 0` via the `fillna(0)` branch
in `_preprocess_results`.  Zero is a *measurement* of "no
responders", but it is being fed to the GP at the same precision as
an FOV with 28 cells.  The GP can’t down-weight it.

*Fix:* either drop these FOVs from the GP fit, or pass per-FOV SE
(`auc_adj_se`, already computed) as a heteroscedastic-noise input
(`gpax.ExactGP` supports a per-point noise input via the `sigma_y`
kwarg in the variant kernels).

In [ ]:
# Drill into the pi=1 dump.
pi1 = df[df["pulse_interval"] == 1]
rest = df[df["pulse_interval"] > 1]
print(
    f"pi = 1   : n_FOVs={len(pi1):3d}, mean auc_adj={pi1[OBJ].mean():.2f} ± "
    f"{pi1[OBJ].std()/np.sqrt(len(pi1)):.2f},  mean frac_responders={pi1['frac_responders'].mean():.2f}"
)
print(
    f"pi > 1   : n_FOVs={len(rest):3d}, mean auc_adj={rest[OBJ].mean():.2f} ± "
    f"{rest[OBJ].std()/np.sqrt(len(rest)):.2f},  mean frac_responders={rest['frac_responders'].mean():.2f}"
)
from scipy.stats import mannwhitneyu

U, p = mannwhitneyu(pi1[OBJ], rest[OBJ], alternative="two-sided")
print(f"Mann-Whitney U (pi=1 vs pi>1): U={U:.0f}, p={p:.3f}")
print(
    "\n=> If p > 0.05, the BO learned nothing from 112 FOVs at pi=1 vs the rest.\n"
    "   That is half the wet-lab budget spent on a non-distinguishable regime."
)

## G — The slide-ready pitch (with what I would and would *not* claim)

### What v15 actually delivered

1. **A working Level-3 ANCOVA pipeline.**  The cell-level
   covariate-adjusted AUC is well-defined, persists across resumed
   runs, and the nuisance fit converges to interpretable slopes.
   The framework is sound; the data is just not strong enough yet.
2. **A free-shape parameterisation that *spans* the shape space.**
   The polynomial-softmax is biologically reasonable (it can
   produce front-loaded, back-loaded, plateau, and bimodal shapes)
   and respects the budget invariant.  The grid spacing is finer
   than was practical with v13/v14’s discrete ramp parameters.
3. **An honest reporting framework.**  The notebook records v14
   `auc_norm`, absolute `auc_above_baseline`, `frac_responders`,
   per-FOV `auc_adj_se`, and the nuisance fit history alongside the
   BO objective.  This is what allows the present pitch-and-critique
   notebook to exist.

### What I would *not* claim in the slide

- ❌ **“The BO found a clear optimum.”**  Section C’s SNR diagnostic
  says the GP signal variance is smaller than its noise variance.
  The “optimum” at $(+3, +2, -3, pi=2)$ is at a grid corner with a
  posterior 95 % CI that overlaps half the search space.
- ❌ **“BO was more sample-efficient than a grid scan.”**  Random
  reordering of the same 217 FOVs reaches the same plateau (Section
  B).  The BO is not paying for itself here.
- ❌ **“The Level-3 adjustment removed nuisance variance.”**  It
  removed *some* (R² = 0.14–0.30 explained at cell level), but the
  pooled slope is biased by between-condition baseline differences
  (F4) and is non-stationary across phases (F5).  Net effect on the
  GP fit is small.
- ❌ **“Pulse_interval matters less than shape.”**  The ARD
  lengthscale on `pulse_interval` is actually *shorter* than on any
  shape axis (Section D).  Among the BO knobs the GP found, `pi` is
  the most informative — but it is still less informative than the
  `n_cells` nuisance covariate.
- ❌ **“This pattern drives functional ERK output.”**  We measured
  KTR cytoplasmic-to-nuclear ratio.  We did not measure ERK target-
  gene induction, proliferation, or any phenotype downstream of
  ERK — only the dynamics of the immediate reporter.

## H — Concrete diffs for the next run

Ranked by expected payoff per unit engineering effort.

### Tier 1 — fix before next plate

1. **Drop `pulse_interval ≤ 2` from the BO grid.**  Half the v15
   budget was spent at $pi = 1$, where the floor-renormalisation
   logic dominates the polynomial shape and the biology is the
   well-known continuous-stim / receptor-internalisation regime.
   Restrict to $pi \in \{3, 4, 6, 8, 11, 15, 20\}$ (7 levels,
   geometrically spaced).  Pair this with a *bigger* budget (8–20 s,
   not 4 s) so even at $pi = 3$ there are 20 pulses with $\sim 400$ ms
   each — well above the floor.  **Expected reclaim: ≈50 % of FOV
   budget redirected to the informative regime.**
2. **Increase cells per FOV.**  Drop the `max_baseline_cnr < 1.00`
   filter (the Level-3 ANCOVA already adjusts for baseline at the
   cell level — the filter is double-correcting), and lift FOV-
   finder `min_cells` to 80.  Median 5 cells / FOV is the single
   biggest contributor to the SNR < 1 finding.  Doubling cells / FOV
   to ∼12 halves the per-FOV SE.  **Expected SNR gain: ≈52 %.**
3. **Switch the polynomial-softmax shape to a 2-parameter biologically
   meaningful basis.**  Two candidates, both with 3×5 = 15-level
   grids per axis (much smaller than the current 7·7·7 = 343):
   - \(\text{amp\_front}, \text{decay\_rate}\) for a decaying-
     exponential envelope: $E_k = E_0 \exp(-\lambda \cdot k / n)$,
     renormalised to the budget.
   - \(\text{tilt}, \text{plateau\_width}\) for a plateau-and-tilt
     basis.
   Either eliminates the corner-degeneracy of the Legendre+softmax
   parameterisation (F2) and shrinks the grid by 10×.

### Tier 2 — fix before next BO design

4. **Switch the ANCOVA to FOV-fixed-effects.**  Replace the pooled
   slope with the within-FOV slope (the regression on residuals of
   baseline and logexpr after removing per-FOV means).  This is the
   correct ANCOVA for a clustered design and eliminates F4.
5. **Freeze the nuisance slopes after a warm-up phase.**  Fit the
   ANCOVA once after 4–5 phases (≈ 600 cells), then *fix* the
   slopes for the remainder of the run and *re-compute* the
   per-FOV `auc_adj` for all earlier phases with the frozen slopes.
   Removes F5 (non-stationarity) for the cost of a one-time
   re-evaluation.
6. **Pass per-FOV `auc_adj_se` to the GP as heteroscedastic noise.**
   `gpax.ExactGP` accepts per-point noise via the `noise_prior_dist`
   mechanism or via a sigma-fixed kernel variant.  Lets the GP
   down-weight FOVs with low cell counts (F11).
7. **Add `phase_id` (or `well_id`) as a categorical fixed effect.**
   Plate drift over the 13-hour run (F8) is presently absorbed into
   GP noise; explicitly modelling it would cut several units of
   variance.

### Tier 3 — do once, gain long-term

8. **Real biological replicates.**  Two plates, different days,
   fresh transfection, pooled with a plate-level random effect.
   The only way to get a credible *across-plate* CI on the
   discovered optimum.
9. **Hierarchical cell~FOV~condition model.**  Stops throwing away
   per-cell heterogeneity and gives proper credible intervals.
   Halves the apparent FOV-level noise.  Same recommendation as v11
   F4; still not done.
10. **Validate the discovered shape with a sustained-ERK functional
    readout.**  e.g. CDK2/CDK4 substrate-based proliferation marker,
    immediate-early-gene induction qPCR, or a parallel FRA1
    fluorescent reporter line.  The KTR pipeline only tells us the
    dynamics of the immediate reporter — not whether the ERK signal
    drives a phenotype.

### Tier 4 — methodological clean-up

11. **Fix the markdown documentation** to match
    `BO_Parameter(spacing=1, bounds=(1,20))` — 20 levels of `pi`, not
    10 (F10).
12. **Track the L1-distance between coefficient-space and exposure-
    space pairs** as a diagnostic alongside ARD.  Whenever two
    distant coefficient triples produce nearly-identical exposure
    profiles, flag them.  Hard cap on "effective dimensionality"
    of the shape parameterisation.
13. **Run an explicit ‘no-stim’ negative control well per phase**
    (currently `WELLS[0]` is sacrificial for starvation monitoring;
    add a second control well that goes through the imaging+
    tracking pipeline without stim).  Lets us calibrate `auc_adj`
    against a hard zero and validate the responder threshold.